### Data source and access
- Dataset: PlantVillage, loaded with TensorFlow Datasets (tfds.load)
- Source: https://www.tensorflow.org/datasets/catalog/plant_village
- TFDS version 1.0.2: 54,303 images, 38 classes
- Packages: tensorflow, tensorflow-datasets, matplotlib, scikit-learn (exact versions in requirements.txt)
- Download is stored in data/tfds, and later runs load from there

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds

print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.19.1
TFDS: 4.9.7
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import os

data_dir = os.path.expanduser("~/Projects/plant-disease-detection/data/tfds")

ds, info = tfds.load(
    "plant_village",
    split="train",
    as_supervised=True,
    with_info=True,
    data_dir=data_dir,
)

print("Total images:", info.splits["train"].num_examples)
print("Total classes:", info.features["label"].num_classes)

2026-09-27 18:12:49.404383: W external/local_xla/xla/tsl/platform/cloud/google_auth_provider.cc:184] All attempts to get a Google authentication bearer token failed, returning an empty token. Retrieving token from files failed with "NOT_FOUND: Could not locate the credentials file.". Retrieving token from GCE failed with "FAILED_PRECONDITION: Error executing an HTTP request: libcurl code 6 meaning 'Could not resolve hostname', error details: Could not resolve host: metadata.google.internal".


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/54303 [00:00<?, ? examples/s]

Shuffling /Users/eloelo/Projects/plant-disease-detection/data/tfds/plant_village/incomplete.FR8PUO_1.0.2/plant…

Dataset plant_village downloaded and prepared to /Users/eloelo/Projects/plant-disease-detection/data/tfds/plant_village/1.0.2. Subsequent calls will reuse this data.
Total images: 54303
Total classes: 38


2026-09-27 19:24:15.894981: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2026-09-27 19:24:15.895106: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-09-27 19:24:15.895129: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
I0000 00:00:1790517255.896323   65032 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1790517255.896814   65032 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
class_names = info.features["label"].names

for i in range(len(class_names)):
    print(i, class_names[i])

0 Apple___Apple_scab
1 Apple___Black_rot
2 Apple___Cedar_apple_rust
3 Apple___healthy
4 Blueberry___healthy
5 Cherry___healthy
6 Cherry___Powdery_mildew
7 Corn___Cercospora_leaf_spot Gray_leaf_spot
8 Corn___Common_rust
9 Corn___healthy
10 Corn___Northern_Leaf_Blight
11 Grape___Black_rot
12 Grape___Esca_(Black_Measles)
13 Grape___healthy
14 Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
15 Orange___Haunglongbing_(Citrus_greening)
16 Peach___Bacterial_spot
17 Peach___healthy
18 Pepper,_bell___Bacterial_spot
19 Pepper,_bell___healthy
20 Potato___Early_blight
21 Potato___healthy
22 Potato___Late_blight
23 Raspberry___healthy
24 Soybean___healthy
25 Squash___Powdery_mildew
26 Strawberry___healthy
27 Strawberry___Leaf_scorch
28 Tomato___Bacterial_spot
29 Tomato___Early_blight
30 Tomato___healthy
31 Tomato___Late_blight
32 Tomato___Leaf_Mold
33 Tomato___Septoria_leaf_spot
34 Tomato___Spider_mites Two-spotted_spider_mite
35 Tomato___Target_Spot
36 Tomato___Tomato_mosaic_virus
37 Tomato___Tomato_Ye

In [4]:
tomato_ids = []

for i in range(len(class_names)):
    if class_names[i].startswith("Tomato"):
        tomato_ids.append(i)

print("Tomato label numbers:", tomato_ids)
print("Number of tomato classes:", len(tomato_ids))

for i in tomato_ids:
    print(i, class_names[i])

Tomato label numbers: [28, 29, 30, 31, 32, 33, 34, 35, 36, 37]
Number of tomato classes: 10
28 Tomato___Bacterial_spot
29 Tomato___Early_blight
30 Tomato___healthy
31 Tomato___Late_blight
32 Tomato___Leaf_Mold
33 Tomato___Septoria_leaf_spot
34 Tomato___Spider_mites Two-spotted_spider_mite
35 Tomato___Target_Spot
36 Tomato___Tomato_mosaic_virus
37 Tomato___Tomato_Yellow_Leaf_Curl_Virus


In [5]:
tomato_ids_tensor = tf.constant(tomato_ids, dtype=tf.int64)

def is_tomato(image, label):
    return tf.reduce_any(label == tomato_ids_tensor)

tomato_ds = ds.filter(is_tomato)

counts = {}
for i in tomato_ids:
    counts[i] = 0

for image, label in tomato_ds:
    counts[int(label)] += 1

total = 0
for i in tomato_ids:
    print(class_names[i], counts[i])
    total += counts[i]

print("Total tomato images:", total)

2026-09-27 19:24:16.154745: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


Tomato___Bacterial_spot 2127
Tomato___Early_blight 1000
Tomato___healthy 1591
Tomato___Late_blight 1908
Tomato___Leaf_Mold 952
Tomato___Septoria_leaf_spot 1771
Tomato___Spider_mites Two-spotted_spider_mite 1676
Tomato___Target_Spot 1404
Tomato___Tomato_mosaic_virus 373
Tomato___Tomato_Yellow_Leaf_Curl_Virus 5357
Total tomato images: 18159


2026-09-27 19:24:18.595023: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [6]:
first_id = tomato_ids[0]

def relabel(image, label):
    return image, label - first_id

tomato_ds = tomato_ds.map(relabel)

tomato_class_names = []
for i in tomato_ids:
    tomato_class_names.append(class_names[i])

for i in range(len(tomato_class_names)):
    print(i, tomato_class_names[i])

0 Tomato___Bacterial_spot
1 Tomato___Early_blight
2 Tomato___healthy
3 Tomato___Late_blight
4 Tomato___Leaf_Mold
5 Tomato___Septoria_leaf_spot
6 Tomato___Spider_mites Two-spotted_spider_mite
7 Tomato___Target_Spot
8 Tomato___Tomato_mosaic_virus
9 Tomato___Tomato_Yellow_Leaf_Curl_Virus


### Classes retained (tomato only)
- Kept only the 10 tomato classes out of 38, filtered by name
- Tomato only, so the model has to learn disease features rather than tell crops apart
- Labels renumbered from 28–37 to 0–9:
  - 0 Bacterial_spot (2,127)
  - 1 Early_blight (1,000)
  - 2 healthy (1,591)
  - 3 Late_blight (1,908)
  - 4 Leaf_Mold (952)
  - 5 Septoria_leaf_spot (1,771)
  - 6 Spider_mites (1,676)
  - 7 Target_Spot (1,404)
  - 8 Tomato_mosaic_virus (373)
  - 9 Tomato_Yellow_Leaf_Curl_Virus (5,357)
- Total: 18,159 images
- The original PlantVillage has 18,160 tomato images; the TFDS copy is missing one Late_blight image

In [7]:
train_ds = None
val_ds = None
test_ds = None

for c in range(len(tomato_class_names)):
    n = counts[tomato_ids[c]]
    n_train = int(n * 0.8)
    n_val = int(n * 0.1)
    n_test = n - n_train - n_val

    class_ds = tomato_ds.filter(lambda image, label: label == c)
    class_ds = class_ds.shuffle(n, seed=42, reshuffle_each_iteration=False)

    class_train = class_ds.take(n_train)
    class_val = class_ds.skip(n_train).take(n_val)
    class_test = class_ds.skip(n_train + n_val)

    if train_ds is None:
        train_ds = class_train
        val_ds = class_val
        test_ds = class_test
    else:
        train_ds = train_ds.concatenate(class_train)
        val_ds = val_ds.concatenate(class_val)
        test_ds = test_ds.concatenate(class_test)

    print(tomato_class_names[c], n, n_train, n_val, n_test)

Tomato___Bacterial_spot 2127 1701 212 214
Tomato___Early_blight 1000 800 100 100
Tomato___healthy 1591 1272 159 160
Tomato___Late_blight 1908 1526 190 192
Tomato___Leaf_Mold 952 761 95 96
Tomato___Septoria_leaf_spot 1771 1416 177 178
Tomato___Spider_mites Two-spotted_spider_mite 1676 1340 167 169
Tomato___Target_Spot 1404 1123 140 141
Tomato___Tomato_mosaic_virus 373 298 37 38
Tomato___Tomato_Yellow_Leaf_Curl_Virus 5357 4285 535 537


### Train / validation / test split
- Split 80/10/10 separately for each class (stratified), shuffled with seed 42
- Result: train 14,522, val 1,812, test 1,825 (total 18,159)
- Did not use the tfds split syntax (train[:80%], train[80%:90%], train[90%:]) because:
  - it slices the full 38-class dataset before the tomato filter
  - it cuts by position, not by class, so each split could end up with a different mix of diseases
- Splitting each class on its own keeps the same disease mix in train, val and test
- Splits saved to data/splits_tfds, so every notebook uses exactly the same images

In [8]:
splits_dir = os.path.expanduser("~/Projects/plant-disease-detection/data/splits_tfds")

train_ds.save(os.path.join(splits_dir, "train"))
val_ds.save(os.path.join(splits_dir, "val"))
test_ds.save(os.path.join(splits_dir, "test"))

print("Saved to", splits_dir)

Saved to /Users/eloelo/Projects/plant-disease-detection/data/splits_tfds


In [9]:
split_names = ["train", "val", "test"]

for split in split_names:
    loaded_ds = tf.data.Dataset.load(os.path.join(splits_dir, split))

    split_counts = []
    for c in range(len(tomato_class_names)):
        split_counts.append(0)

    for image, label in loaded_ds:
        split_counts[int(label)] += 1

    print(split, "total:", sum(split_counts))
    for c in range(len(tomato_class_names)):
        print("  ", tomato_class_names[c], split_counts[c])

print("Image shape:", image.shape, image.dtype)

2026-09-27 19:25:39.503073: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


train total: 14522
   Tomato___Bacterial_spot 1701
   Tomato___Early_blight 800
   Tomato___healthy 1272
   Tomato___Late_blight 1526
   Tomato___Leaf_Mold 761
   Tomato___Septoria_leaf_spot 1416
   Tomato___Spider_mites Two-spotted_spider_mite 1340
   Tomato___Target_Spot 1123
   Tomato___Tomato_mosaic_virus 298
   Tomato___Tomato_Yellow_Leaf_Curl_Virus 4285
val total: 1812
   Tomato___Bacterial_spot 212
   Tomato___Early_blight 100
   Tomato___healthy 159
   Tomato___Late_blight 190
   Tomato___Leaf_Mold 95
   Tomato___Septoria_leaf_spot 177
   Tomato___Spider_mites Two-spotted_spider_mite 167
   Tomato___Target_Spot 140
   Tomato___Tomato_mosaic_virus 37
   Tomato___Tomato_Yellow_Leaf_Curl_Virus 535
test total: 1825
   Tomato___Bacterial_spot 214
   Tomato___Early_blight 100
   Tomato___healthy 160
   Tomato___Late_blight 192
   Tomato___Leaf_Mold 96
   Tomato___Septoria_leaf_spot 178
   Tomato___Spider_mites Two-spotted_spider_mite 169
   Tomato___Target_Spot 141
   Tomato___Tomato

2026-09-27 19:25:39.780853: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [10]:
import hashlib

hash_info = {}

for split in split_names:
    loaded_ds = tf.data.Dataset.load(os.path.join(splits_dir, split))
    for image, label in loaded_ds:
        h = hashlib.md5(image.numpy().tobytes()).hexdigest()
        if h not in hash_info:
            hash_info[h] = []
        hash_info[h].append((split, int(label)))

total_images = 0
for h in hash_info:
    total_images += len(hash_info[h])

duplicate_groups = 0
across_splits = 0
different_labels = 0

for h in hash_info:
    entries = hash_info[h]
    if len(entries) > 1:
        duplicate_groups += 1
        splits_seen = set()
        labels_seen = set()
        for split, label in entries:
            splits_seen.add(split)
            labels_seen.add(label)
        if len(splits_seen) > 1:
            across_splits += 1
        if len(labels_seen) > 1:
            different_labels += 1

print("Total images:", total_images)
print("Unique images:", len(hash_info))
print("Duplicate groups:", duplicate_groups)
print("Duplicates across splits:", across_splits)
print("Duplicates with different labels:", different_labels)

Total images: 18159
Unique images: 18145
Duplicate groups: 14
Duplicates across splits: 4
Duplicates with different labels: 0


### Data-quality checks
- Split counts checked after saving: train 14,522, val 1,812, test 1,825, matching the split cell
- Duplicate check (pixel-by-pixel hash of every image):
  - 18,159 images, 18,145 unique, so 14 exact duplicate pairs in the source data
  - All 14 pairs have the same label, so no labelling conflicts
  - 4 pairs are split across train and val; none reach the test set
- Kept the duplicates: the effect on val is about 0.2% (4 of 1,812 images), and the test set is clean

### Resizing and normalising
- Saved images are 256 x 256 x 3, pixel values 0–255 (uint8), as they come from TFDS
- Resizing to 224 x 224 happens when each model notebook loads the splits (02, 03, 04)
- Normalising depends on the model:
  - Baseline CNN: divide by 255, so pixels are 0–1
  - MobileNetV2: Rescaling layer inside the model, so pixels are -1 to 1
  - ResNet50: ResNet50's own preprocess_input
- The splits are saved without normalising, so each model can prepare the pixels its own way